# Task 4 — Track eggs in the video

This notebook applies the final Task 2 segmentation method to the provided egg-spinning video. It labels detected eggs and tries to keep each label consistent from frame to frame. The output is an annotated video plus a CSV of positions and sizes over time.

**Before running:** make the Task 2 `task2_results` folder available, including `final_config.json`. The video is found automatically if it is beside the notebook; otherwise set `EGG_VIDEO` in the first code cell. Install SciPy if it is not already available; it is used to match detections between frames.

In [ ]:
from pathlib import Path
import os, json, math, warnings
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path.cwd()
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
from scipy.optimize import linear_sum_assignment
VIDEO_PATH = Path(os.environ.get("EGG_VIDEO", "")) if os.environ.get("EGG_VIDEO") else None
if VIDEO_PATH is None:
    candidates = list(ROOT.glob("*.mp4")) + list(ROOT.rglob("*.mp4"))
    if not candidates: raise FileNotFoundError("Set EGG_VIDEO to the provided mp4 file.")
    VIDEO_PATH = candidates[0]
TASK2_DIR = Path(os.environ.get("TASK2_DIR", "")) if os.environ.get("TASK2_DIR") else None
if TASK2_DIR is None:
    for search_root in [ROOT, Path("/kaggle/input"), Path("/kaggle/working")]:
        if search_root.exists():
            found = sorted(search_root.rglob("task2_results"))
            if found: TASK2_DIR = found[0]; break
if TASK2_DIR is None or not (TASK2_DIR/"final_config.json").exists():
    raise FileNotFoundError("Need Task 2 task2_results/final_config.json. Run Task 2 and attach its output folder.")
with open(TASK2_DIR/"final_config.json", encoding="utf-8") as f: cfg = json.load(f)
# JSON stores tuple-valued parameters as lists; the ellipse code consumes sequences either way.
print("Video:", VIDEO_PATH, "\nTask 2 config:", TASK2_DIR/"final_config.json")


## Use the same segmentation as Task 2

The next cell reuses the segmentation functions from the final Task 2 notebook. The settings are loaded from the saved `final_config.json`, so the video uses the same chosen method.

In [ ]:
# These are the segmentation functions from Task2/task2_ThirdTry.ipynb (cell 9).
# It expects cv2, numpy as np, math, and SEED defined above.
BASE_CFG = dict(
    method="otsu", proc_size=320, blur=5, open_k=7, min_area=0.008, max_area=0.8, max_eggs=3,
    hsv_s=60, hsv_v=140, canny_lo=40, canny_hi=120, canny_close=5, rect_margin=0.12,
    gens=("otsu", "hsv", "border"), score_thr=0.6,
    w_ell=0.35, w_sol=0.15, w_ratio=0.15, w_con=0.25, w_white=0.10, w_border=0.6,
    refine=None, gc_iters=4, gc_core=0.35, gc_band=0.25, ellipse_blend=None,
    # ---- phase 2: ellipse search (defaults = the naive first version) ----
    search_size=160, center_step=8, a_fracs=(0.07, 0.10, 0.14, 0.19, 0.25, 0.32, 0.40),
    ratios=(0.6, 0.75, 0.9), theta_step=30, d=3.0, w_mean=0.5, pct=25, cap=35.0,
    dark_w=0.7, w_int=0.0, w_size=0.0, n_refine=8,
    extra_ratio=0.7, extra_area=0.0, extra_dlab=1e9, extra_overlap=1.0,
    ell_refine=None, eg_size=160, eg_core=0.55, eg_out=1.35, eg_iters=3, eg_guard=0.6, eg_union=0.0, eg_clean=1,
)

def odd(k):
    k = int(max(1, round(k)))
    return k if k % 2 == 1 else k + 1

def ell_kernel(k):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (odd(k), odd(k)))

def to_work(img, size):
    h, w = img.shape[:2]
    s = min(1.0, size / max(h, w))
    if s < 1.0:
        return cv2.resize(img, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA)
    return img.copy()

def fill_holes(mask):
    m = mask.astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(m)
    cv2.drawContours(out, cnts, -1, 1, thickness=-1)
    return out.astype(bool)

def components(mask, min_area_px):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    return [lab == i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area_px]

def postprocess(mask, cfg):
    h, w = mask.shape
    k = ell_kernel(cfg["open_k"])
    m = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, k)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    m = fill_holes(m)
    comps = sorted(components(m, cfg["min_area"] * h * w), key=lambda c: -c.sum())[: cfg["max_eggs"]]
    out = np.zeros((h, w), bool)
    for c in comps: out |= c
    return out

# ---------------- foreground generators ----------------
def gen_otsu(iw, cfg):
    g = cv2.GaussianBlur(cv2.cvtColor(iw, cv2.COLOR_RGB2GRAY), (odd(cfg["blur"]),) * 2, 0)
    t, _ = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    b = g > t
    return [b, ~b]

def gen_hsv(iw, cfg):
    hsv = cv2.cvtColor(iw, cv2.COLOR_RGB2HSV)
    s, v = hsv[..., 1], hsv[..., 2]
    return [(s < cfg["hsv_s"]) & (v > cfg["hsv_v"]),
            (s < cfg["hsv_s"] * 0.7) & (v > cfg["hsv_v"] * 0.7)]

def gen_border(iw, cfg, frac=0.08):
    lab = cv2.cvtColor(iw, cv2.COLOR_RGB2LAB).astype(np.float32)
    h, w = lab.shape[:2]; b = max(2, int(frac * min(h, w)))
    ring = np.zeros((h, w), bool)
    ring[:b] = True; ring[-b:] = True; ring[:, :b] = True; ring[:, -b:] = True
    bg = np.median(lab[ring], axis=0)
    d = np.linalg.norm(lab - bg, axis=2)
    d8 = np.clip(d / max(d.max(), 1e-6) * 255, 0, 255).astype(np.uint8)
    d8 = cv2.GaussianBlur(d8, (odd(cfg["blur"]),) * 2, 0)
    t, _ = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return [d8 > t]

def gen_canny(iw, cfg):
    g = cv2.GaussianBlur(cv2.cvtColor(iw, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    e = cv2.Canny(g, cfg["canny_lo"], cfg["canny_hi"])
    e = cv2.morphologyEx(e, cv2.MORPH_CLOSE, ell_kernel(cfg["canny_close"]))
    return [e == 0]          # regions enclosed by edges

GENS = {"otsu": gen_otsu, "hsv": gen_hsv, "border": gen_border, "canny": gen_canny}

def grabcut_rect(iw, cfg):
    h, w = iw.shape[:2]
    mx, my = int(cfg["rect_margin"] * w), int(cfg["rect_margin"] * h)
    mask = np.zeros((h, w), np.uint8)
    bgd = np.zeros((1, 65)); fgd = np.zeros((1, 65))
    cv2.setRNGSeed(SEED)     # GrabCut's GMM init is random; fix it so trials are reproducible
    cv2.grabCut(cv2.cvtColor(iw, cv2.COLOR_RGB2BGR), mask, (mx, my, w - 2 * mx, h - 2 * my),
                bgd, fgd, cfg["gc_iters"], cv2.GC_INIT_WITH_RECT)
    return (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD)

# ---------------- candidate generation + egg-likeness scoring ----------------
def candidate_components(iw, cfg):
    h, w = iw.shape[:2]
    k = ell_kernel(cfg["open_k"])
    out = []
    for g in cfg["gens"]:
        for b in GENS[g](iw, cfg):
            m = cv2.morphologyEx(b.astype(np.uint8), cv2.MORPH_OPEN, k)
            m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
            for c in components(m, cfg["min_area"] * h * w):
                c = fill_holes(c)
                if c.sum() / (h * w) <= cfg["max_area"]:
                    out.append(c)
    return out

def score_component(comp, lab, cfg):
    h, w = comp.shape
    m = comp.astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts: return None
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5: return None
    area = cv2.contourArea(c)
    if area < 1: return None
    hull = cv2.contourArea(cv2.convexHull(c))
    solidity = area / max(hull, 1e-6)
    try:
        ell = cv2.fitEllipse(c)
    except cv2.error:
        return None
    em = np.zeros((h, w), np.uint8); cv2.ellipse(em, ell, 1, -1); em = em.astype(bool)
    ell_iou = (em & comp).sum() / max((em | comp).sum(), 1)
    ax = sorted(ell[1]); ratio = ax[1] / max(ax[0], 1e-6)
    ratio_score = math.exp(-((math.log(max(ratio, 1.0)) - math.log(1.3)) ** 2) / (2 * 0.35 ** 2))
    peri = cv2.arcLength(c, True)
    touch = comp[0, :].sum() + comp[-1, :].sum() + comp[:, 0].sum() + comp[:, -1].sum()
    border_frac = min(1.0, touch / max(peri, 1.0))
    r = math.sqrt(area / math.pi)
    ring = (cv2.dilate(m, ell_kernel(max(5, 0.3 * r))) > 0) & ~comp
    contrast = np.linalg.norm(lab[comp].mean(0) - lab[ring].mean(0)) if ring.sum() >= 10 else 0.0
    contrast_s = min(1.0, contrast / 40.0)
    chroma = np.sqrt(((lab[comp][:, 1:] - 128.0) ** 2).sum(1)).mean()
    white_s = float(np.clip(1 - chroma / 40.0, 0, 1))
    return (cfg["w_ell"] * ell_iou + cfg["w_sol"] * solidity + cfg["w_ratio"] * ratio_score
            + cfg["w_con"] * contrast_s + cfg["w_white"] * white_s - cfg["w_border"] * border_frac)

def select_eggs(cands, lab, cfg):
    scored = []
    for c in cands:
        s = score_component(c, lab, cfg)
        if s is not None: scored.append((s, c))
    scored.sort(key=lambda t: -t[0])
    sel = []
    for s, c in scored:
        if len(sel) >= cfg["max_eggs"]: break
        overlap = any((c & k).sum() / max(min(c.sum(), k.sum()), 1) > 0.3 for _, k in sel)
        if overlap: continue
        if not sel or s >= cfg["score_thr"]:
            sel.append((s, c))
    return sel

# ---------------- refinement ----------------
def grabcut_refine(iw, union, cfg):
    h, w = union.shape
    area = union.sum()
    if area < 50: return union
    r = math.sqrt(area / math.pi)
    u8 = union.astype(np.uint8)
    sure = cv2.erode(u8, ell_kernel(cfg["gc_core"] * r * 2)) > 0
    band = cv2.dilate(u8, ell_kernel(cfg["gc_band"] * r * 2)) > 0
    if sure.sum() < 20: sure = union
    mask = np.full((h, w), cv2.GC_BGD, np.uint8)
    mask[band] = cv2.GC_PR_BGD
    mask[union] = cv2.GC_PR_FGD
    mask[sure] = cv2.GC_FGD
    bgd = np.zeros((1, 65)); fgd = np.zeros((1, 65))
    cv2.setRNGSeed(SEED)
    try:
        cv2.grabCut(cv2.cvtColor(iw, cv2.COLOR_RGB2BGR), mask, None, bgd, fgd, cfg["gc_iters"], cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        return union
    return (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD)

def ellipse_blend(mask, thr):
    out = np.zeros_like(mask)
    for c in components(mask, 20):
        used = c
        cnts, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cc = max(cnts, key=cv2.contourArea)
        if len(cc) >= 5:
            try:
                em = np.zeros(c.shape, np.uint8); cv2.ellipse(em, cv2.fitEllipse(cc), 1, -1); em = em.astype(bool)
                if (em & c).sum() / max((em | c).sum(), 1) >= thr: used = em
            except cv2.error:
                pass
        out |= used
    return out

# ---------------- full pipeline ----------------
def segment(img, cfg):
    H, W = img.shape[:2]
    if cfg["method"] == "ellipse":
        return segment_ellipse(img, cfg)
    iw = to_work(img, cfg["proc_size"])
    m = cfg["method"]
    if m == "otsu":           raw = gen_otsu(iw, cfg)[0]
    elif m == "hsv":          raw = gen_hsv(iw, cfg)[0]
    elif m == "border":       raw = gen_border(iw, cfg)[0]
    elif m == "grabcut_rect": raw = grabcut_rect(iw, cfg)
    elif m == "candidates":
        lab = cv2.cvtColor(iw, cv2.COLOR_RGB2LAB).astype(np.float32)
        sel = select_eggs(candidate_components(iw, cfg), lab, cfg)
        raw = np.zeros(iw.shape[:2], bool)
        for _, c in sel: raw |= c
    else:
        raise ValueError(m)
    pp = postprocess(raw, cfg)
    if cfg.get("refine") == "grabcut" and pp.any():
        pp = postprocess(grabcut_refine(iw, pp, cfg), cfg)
    if cfg.get("ellipse_blend") and pp.any():
        pp = ellipse_blend(pp, cfg["ellipse_blend"])
    return cv2.resize(pp.astype(np.uint8) * 255, (W, H), interpolation=cv2.INTER_LINEAR) > 127

# =====================================================================
# PHASE 2: ellipse search ("find the egg-shaped outline"), then optional GrabCut inside it
# =====================================================================
def ellipse_pts(cx, cy, a, b, th, K=36):
    # points on N ellipses (N,K) and their outward unit normals
    t = np.linspace(0, 2 * np.pi, K, endpoint=False)[None, :]
    ct, st = np.cos(t), np.sin(t)
    c, s_ = np.cos(th)[:, None], np.sin(th)[:, None]
    ex, ey = a[:, None] * ct, b[:, None] * st
    x = cx[:, None] + c * ex - s_ * ey
    y = cy[:, None] + s_ * ex + c * ey
    nx0, ny0 = ct / a[:, None], st / b[:, None]
    nx, ny = c * nx0 - s_ * ny0, s_ * nx0 + c * ny0
    nn = np.sqrt(nx ** 2 + ny ** 2) + 1e-9
    return x, y, nx / nn, ny / nn

def draw_ellipse(shape, e):
    m = np.zeros(shape, np.uint8)
    cv2.ellipse(m, ((float(e[0]), float(e[1])), (float(2 * e[2]), float(2 * e[3])), float(np.rad2deg(e[4]))), 1, -1)
    return m.astype(bool)

def grad_mag(lab):
    g = np.zeros(lab.shape[:2], np.float32)
    for ch in range(3):
        gx = cv2.Sobel(lab[..., ch], cv2.CV_32F, 1, 0, ksize=3) / 8.0
        gy = cv2.Sobel(lab[..., ch], cv2.CV_32F, 0, 1, ksize=3) / 8.0
        g += gx ** 2 + gy ** 2
    return np.sqrt(g)

def sample(img2d, x, y):
    H, W = img2d.shape[:2]
    return img2d[np.clip(np.rint(y), 0, H - 1).astype(int), np.clip(np.rint(x), 0, W - 1).astype(int)]

def score_ellipses(lab, gm, cx, cy, a, b, th, P):
    # Egg-outline score = colour contrast across the boundary (inside vs outside, 36 points)
    #                     - penalty for texture inside the ellipse (eggs are smooth).
    H, W = lab.shape[:2]; d = P["d"]
    x, y, nx, ny = ellipse_pts(cx, cy, a, b, th, 36)
    di = np.minimum(d, 0.35 * np.minimum(a, b))[:, None]
    xi, yi, xo, yo = x - di * nx, y - di * ny, x + d * nx, y + d * ny
    valid = (xo >= 0) & (xo <= W - 1) & (yo >= 0) & (yo <= H - 1) & (xi >= 0) & (xi <= W - 1) & (yi >= 0) & (yi <= H - 1)
    li, lo = sample(lab, xi, yi), sample(lab, xo, yo)
    dist = np.linalg.norm(li - lo, axis=2)
    bright = np.where(li[..., 0] >= lo[..., 0], 1.0, P["dark_w"])      # eggs are usually brighter than surroundings
    dist = np.where(valid, np.minimum(dist, P["cap"]) * bright, 0.0)
    contrast = P["w_mean"] * dist.mean(1) + (1 - P["w_mean"]) * np.percentile(dist, P["pct"], axis=1)
    score = contrast
    if P["w_int"] > 0:
        tex = 0.0
        for sc_ in (0.4, 0.75):
            xx, yy, _, _ = ellipse_pts(cx, cy, a * sc_, b * sc_, th, 20)
            tex = tex + sample(gm, xx, yy).mean(1) / 2
        score = score - P["w_int"] * tex * 2 * d
    if P["w_size"] > 0:
        score = score + P["w_size"] * np.sqrt(a * b) / min(H, W) * 20
    return score

def search_ellipses(lab, gm, P, exclude):
    H, W = lab.shape[:2]; m = min(H, W); step = P["center_step"]
    cxs = np.arange(step, W - step + 1, step); cys = np.arange(step, H - step + 1, step)
    G = np.array(np.meshgrid(cxs, cys, m * np.array(P["a_fracs"]), np.array(P["ratios"]),
                             np.deg2rad(np.arange(0, 180, P["theta_step"])), indexing="ij")).reshape(5, -1)
    cx, cy, a, rr, th = G; b = a * rr
    if exclude is not None:
        keep = ~exclude[np.clip(cy.astype(int), 0, H - 1), np.clip(cx.astype(int), 0, W - 1)]
        cx, cy, a, b, th = cx[keep], cy[keep], a[keep], b[keep], th[keep]
    if len(cx) == 0: return []
    sc = np.empty(len(cx), np.float32)
    for i in range(0, len(cx), 15000):
        sl = slice(i, i + 15000); sc[sl] = score_ellipses(lab, gm, cx[sl], cy[sl], a[sl], b[sl], th[sl], P)
    order = np.argsort(-sc)[: P["n_refine"]]
    return [np.array([cx[i], cy[i], a[i], b[i], th[i]]) for i in order]

def refine_ellipse(lab, gm, e, P, iters=3):
    # local coordinate search around a coarse grid hit (centre, axes, angle), step halves each round
    f = lambda C: score_ellipses(lab, gm, C[:, 0], C[:, 1], C[:, 2], C[:, 3], C[:, 4], P)
    e = e.copy(); cur = float(f(e[None])[0]); steps = np.array([3., 3., 3., 3., np.deg2rad(10)])
    for _ in range(iters):
        improved = True
        while improved:
            improved = False; C = []
            for k in range(5):
                for sg in (-1, 1):
                    c = e.copy(); c[k] += sg * steps[k]
                    if c[2] >= 4 and c[3] >= 4: C.append(c)
            C = np.array(C); sc = f(C); j = int(np.argmax(sc))
            if sc[j] > cur + 1e-4: cur, e, improved = float(sc[j]), C[j], True
        steps = steps / 2
    if e[3] > e[2]: e = np.array([e[0], e[1], e[3], e[2], e[4] + np.pi / 2])
    return cur, e

SEARCH_KEYS = ("search_size", "center_step", "a_fracs", "ratios", "theta_step", "d", "w_mean", "pct", "cap",
               "dark_w", "w_int", "w_size", "n_refine")
_DET_CACHE = {}

def get_detection(img, cfg):
    # up to 3 successive best ellipses (each search excludes centres inside earlier ones); cached per image+search params
    key = (id(img),) + tuple(str(cfg[k]) for k in SEARCH_KEYS)
    if key in _DET_CACHE: return _DET_CACHE[key]
    P = {k: cfg[k] for k in SEARCH_KEYS}
    H, W = img.shape[:2]; s = P["search_size"] / max(H, W)
    small = cv2.GaussianBlur(cv2.resize(img, (int(round(W * s)), int(round(H * s))), interpolation=cv2.INTER_AREA), (0, 0), 1.2)
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).astype(np.float32); gm = grad_mag(lab); h, w = lab.shape[:2]
    found, excl = [], None
    for _ in range(3):
        cands = search_ellipses(lab, gm, P, excl)
        if not cands: break
        sc, e = max((refine_ellipse(lab, gm, c, P) for c in cands), key=lambda t: t[0])
        found.append((sc, e))
        grown = cv2.dilate(draw_ellipse((h, w), e).astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
        excl = grown if excl is None else (excl | grown)
    _DET_CACHE[key] = (small, found, s)
    return _DET_CACHE[key]

def accept_eggs(small, found, cfg):
    # first (best) ellipse is always an egg; further ones must look like *another egg*
    if not found: return []
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).astype(np.float32); h, w = lab.shape[:2]
    def mean_lab(e):
        e2 = e.copy(); e2[2:4] *= 0.8; return lab[draw_ellipse((h, w), e2)].mean(0)
    out = [found[0]]; c0 = mean_lab(found[0][1]); a0 = math.pi * found[0][1][2] * found[0][1][3]
    masks = [draw_ellipse((h, w), found[0][1])]
    for sc, e in found[1: cfg["max_eggs"]]:
        m = draw_ellipse((h, w), e); area = m.sum()
        overlap = max((m & k).sum() / max(min(area, k.sum()), 1) for k in masks)
        ok = (sc >= cfg["extra_ratio"] * found[0][0] and area >= cfg["extra_area"] * a0
              and overlap < cfg["extra_overlap"] and np.linalg.norm(mean_lab(e) - c0) < cfg["extra_dlab"])
        if not ok: break
        out.append((sc, e)); masks.append(m)
    return out

def egg_mask(gimg, e, cfg):
    # mask for one egg at GrabCut resolution: the ellipse itself, or GrabCut seeded inside/around it
    h, w = gimg.shape[:2]; em = draw_ellipse((h, w), e)
    if cfg["ell_refine"] != "grabcut": return em
    core = e.copy(); core[2:4] *= cfg["eg_core"]; outer = e.copy(); outer[2:4] *= cfg["eg_out"]
    mask = np.full((h, w), cv2.GC_BGD, np.uint8)
    mask[draw_ellipse((h, w), outer)] = cv2.GC_PR_BGD
    mask[em] = cv2.GC_PR_FGD
    mask[draw_ellipse((h, w), core)] = cv2.GC_FGD
    bgd = np.zeros((1, 65)); fgd = np.zeros((1, 65)); cv2.setRNGSeed(SEED)
    try:
        cv2.grabCut(cv2.cvtColor(gimg, cv2.COLOR_RGB2BGR), mask, None, bgd, fgd, cfg["eg_iters"], cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        return em
    res = (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD)
    if cfg["eg_clean"] > 1:
        k = ell_kernel(cfg["eg_clean"]); u = res.astype(np.uint8)
        res = fill_holes(cv2.morphologyEx(cv2.morphologyEx(u, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k))
    if (res & em).sum() / max((res | em).sum(), 1) < cfg["eg_guard"]:      # GrabCut leaked: fall back to the ellipse
        return em
    if cfg["eg_union"] > 0:
        e2 = e.copy(); e2[2:4] *= cfg["eg_union"]; res = res | draw_ellipse((h, w), e2)
    return res

def segment_ellipse(img, cfg):
    H, W = img.shape[:2]
    small, found, s = get_detection(img, cfg)
    acc = accept_eggs(small, found, cfg)
    gs = min(1.0, cfg["eg_size"] / max(H, W))
    gimg = cv2.resize(img, (int(round(W * gs)), int(round(H * gs))), interpolation=cv2.INTER_AREA) if gs < 1 else img
    ratio = (gimg.shape[1] / small.shape[1])
    h, w = gimg.shape[:2]; union = np.zeros((h, w), bool)
    for sc, e in acc:
        eg = np.array([e[0] * ratio, e[1] * ratio, e[2] * ratio, e[3] * ratio, e[4]])
        union |= egg_mask(gimg, eg, cfg)
    return cv2.resize(union.astype(np.uint8) * 255, (W, H), interpolation=cv2.INTER_LINEAR) > 127

## Find eggs and link them between frames

Each connected mask region is treated as one egg. The notebook compares the centre of each detection with the centres from the previous frame and chooses the closest sensible matches. A detection that cannot be matched starts a new track. A track can survive a few missed frames, which helps with brief occlusions.

In [ ]:
MAX_MISSED = 8
MAX_CENTER_DISTANCE = 0.12  # fraction of frame diagonal
MIN_COMPONENT_AREA = 0.001  # fraction of image area
OUT_DIR = Path(os.environ.get("TASK4_OUT_DIR", ROOT / "task4_output")); OUT_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened(): raise RuntimeError(f"Could not open video: {VIDEO_PATH}")
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer = cv2.VideoWriter(str(OUT_DIR/"tracked_eggs.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width,height))
tracks, next_id, records = {}, 1, []
max_dist = MAX_CENTER_DISTANCE * math.hypot(width,height)
frame_idx = 0
while True:
    ok, frame_bgr = cap.read()
    if not ok: break
    frame_idx += 1
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mask = segment(frame_rgb, cfg)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    detections = []
    for lab in range(1,n):
        x,y,w,h,area = stats[lab]
        if area < MIN_COMPONENT_AREA*width*height: continue
        cx,cy = centroids[lab]
        detections.append({"bbox":(int(x),int(y),int(w),int(h)),"center":np.array([cx,cy],float),"area":int(area)})
    active_ids = list(tracks.keys())
    matched_t, matched_d = set(), set()
    if active_ids and detections:
        cost = np.array([[np.linalg.norm(tracks[tid]["center"]-d["center"]) for d in detections] for tid in active_ids])
        rr,cc = linear_sum_assignment(cost)
        for r,c in zip(rr,cc):
            if cost[r,c] <= max_dist:
                tid=active_ids[r]; tracks[tid].update(center=detections[c]["center"], bbox=detections[c]["bbox"], missed=0)
                detections[c]["track_id"]=tid; matched_t.add(tid); matched_d.add(c)
    for tid in active_ids:
        if tid not in matched_t: tracks[tid]["missed"] += 1
    for j,d in enumerate(detections):
        if j not in matched_d:
            tid=next_id; next_id+=1; d["track_id"]=tid
            tracks[tid]={"center":d["center"],"bbox":d["bbox"],"missed":0}
    tracks={tid:t for tid,t in tracks.items() if t["missed"] <= MAX_MISSED}
    annotated=frame_bgr.copy()
    for d in detections:
        tid=d["track_id"]; x,y,w,h=d["bbox"]; cx,cy=d["center"].astype(int)
        cv2.rectangle(annotated,(x,y),(x+w,y+h),(0,255,0),2)
        cv2.circle(annotated,(cx,cy),4,(0,0,255),-1)
        cv2.putText(annotated,f"Egg {tid}",(x,max(20,y-8)),cv2.FONT_HERSHEY_SIMPLEX,.7,(0,255,0),2,cv2.LINE_AA)
        records.append({"frame":frame_idx,"time_seconds":(frame_idx-1)/fps,"track_id":tid,"center_x":float(cx),"center_y":float(cy),"x":x,"y":y,"width":w,"height":h,"area_pixels":d["area"]})
    writer.write(annotated)
    if frame_idx % 100 == 0: print(f"Processed {frame_idx}/{frame_count} frames")
cap.release(); writer.release()
tracks_df=pd.DataFrame(records)
tracks_df.to_csv(OUT_DIR/"tracks.csv",index=False)
print("Frames processed:",frame_idx,"| detections:",len(tracks_df),"| track IDs:",sorted(tracks_df.track_id.unique()) if len(tracks_df) else [])
print("Video:", (OUT_DIR/"tracked_eggs.mp4").resolve())


## Review the tracks

The table shows how many frames each ID was visible, and the plot shows its path. IDs may split or swap if eggs overlap, disappear for a long time, or move too far between frames. The video has scene changes, so an ID should not be read as a guarantee of the same physical egg across a cut.

In [ ]:
if not tracks_df.empty:
    counts=tracks_df.groupby("track_id").size().sort_values(ascending=False)
    display(counts.rename("observed frames").to_frame())
    plt.figure(figsize=(10,5))
    for tid,group in tracks_df.groupby("track_id"):
        plt.plot(group.center_x, group.center_y, marker=".", ms=2, lw=1, label=f"Egg {tid}")
    plt.gca().invert_yaxis(); plt.xlabel("x (pixels)"); plt.ylabel("y (pixels)"); plt.title("Tracked centroid paths"); plt.legend(ncol=2, fontsize=8); plt.grid(alpha=.2); plt.show()
else: print("No egg components passed the configured area filter. Adjust segmentation or MIN_COMPONENT_AREA.")
